In [9]:
import os
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras

In [10]:
np.random.seed(42)
tf.random.set_seed(42)

# Chuẩn bị dữ liệu


In [24]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

data_dir = "../data/FER2013/train"
input_size = (48, 48)
val_frac = 0.1
batch_size = 32

data_augmentor = ImageDataGenerator(rotation_range=10,
                                    width_shift_range=0.1,
                                    height_shift_range=0.1,
                                    zoom_range=0.15,
                                    horizontal_flip=True,
                                    samplewise_center=True,
                                    samplewise_std_normalization=True,
                                    validation_split=val_frac)

train_generator = data_augmentor.flow_from_directory(data_dir, 
                                                     target_size=input_size, 
                                                     batch_size=batch_size, 
                                                     color_mode='grayscale',
                                                     shuffle=True, 
                                                     subset="training")
val_generator = data_augmentor.flow_from_directory(data_dir, 
                                                   target_size=input_size, 
                                                   batch_size=batch_size, 
                                                   color_mode='grayscale',
                                                   subset="validation")

Found 25841 images belonging to 7 classes.
Found 2868 images belonging to 7 classes.


In [25]:
train_generator.class_indices

{'angry': 0,
 'disgust': 1,
 'fear': 2,
 'happy': 3,
 'neutral': 4,
 'sad': 5,
 'surprise': 6}

# Xây dựng mô hình


In [26]:
num_class = len(train_generator.class_indices)
input_shape = (48,48,1)

model = keras.models.Sequential([
    keras.Input(shape=input_shape),
    
    #BLOCK 1
    keras.layers.Conv2D(32, kernel_size=(3,3), activation="relu", padding='same'),
    keras.layers.Conv2D(32, kernel_size=(3,3), activation="relu", padding='same'),
    keras.layers.MaxPooling2D(pool_size=(2,2)),
    keras.layers.Dropout(0.25),
    
    #BLOCK 2
    keras.layers.Conv2D(64, kernel_size=(3,3), activation="relu", padding='same'),
    keras.layers.Conv2D(64, kernel_size=(3,3), activation="relu", padding='same'),
    keras.layers.MaxPooling2D(pool_size=(2,2)),
    keras.layers.Dropout(0.25),

    #BLOCK 3
    keras.layers.Conv2D(128, kernel_size=(3,3), activation="relu", padding='same'),
    keras.layers.Conv2D(128, kernel_size=(3,3), activation="relu", padding='same'),
    keras.layers.MaxPooling2D(pool_size=(2,2)),
    keras.layers.Dropout(0.25),

    #FINAL
    keras.layers.Flatten(),
    keras.layers.Dense(512, activation='relu'),
    keras.layers.Dropout(0.5),
    keras.layers.Dense(num_class, activation='softmax'),
])

model.compile(optimizer='adam',
             loss='categorical_crossentropy',
             metrics=['accuracy'])
model.summary()

Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_21 (Conv2D)              │ (None, 48, 48, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_22 (Conv2D)              │ (None, 48, 48, 32)     │         9,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_12 (MaxPooling2D) │ (None, 24, 24, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_16 (Dropout)            │ (None, 24, 24, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_23 (Conv2D)              │ (None, 24, 24, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_24 (Conv2D)              │ (None, 24, 24, 64)     │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_13 (MaxPooling2D) │ (None, 12, 12, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_17 (Dropout)            │ (None, 12, 12, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_25 (Conv2D)              │ (None, 12, 12, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_26 (Conv2D)              │ (None, 12, 12, 128)    │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_14 (MaxPooling2D) │ (None, 6, 6, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_18 (Dropout)            │ (None, 6, 6, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_4 (Flatten)             │ (None, 4608)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 512)            │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_19 (Dropout)            │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 7)              │         3,591 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,649,831 (10.11 MB)

 Trainable params: 2,649,831 (10.11 MB)

 Non-trainable params: 0 (0.00 B)

# Huấn luyện mô hình


In [28]:
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau

checkpoint = ModelCheckpoint(
    filepath='CNN_ep{epoch:02d}.h5',
    save_weights_only=False,
    save_best_only=False,
    monitor='val_loss',
    verbose=1
)

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True,
    verbose=1,
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=5,
    min_lr=1e-7,
    verbose=1
)

history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=50,
    callbacks=[checkpoint, early_stop, reduce_lr]
)

Epoch 1/50
808/808 ━━━━━━━━━━━━━━━━━━━━ 0s 79ms/step - accuracy: 0.4056 - loss: 1.5273
Epoch 1: saving model to CNN_ep01.h5


808/808 ━━━━━━━━━━━━━━━━━━━━ 68s 84ms/step - accuracy: 0.4204 - loss: 1.4963 - val_accuracy: 0.4648 - val_loss: 1.4074 - learning_rate: 0.0010
Epoch 2/50
808/808 ━━━━━━━━━━━━━━━━━━━━ 0s 76ms/step - accuracy: 0.4531 - loss: 1.4244
Epoch 2: saving model to CNN_ep02.h5


808/808 ━━━━━━━━━━━━━━━━━━━━ 64s 79ms/step - accuracy: 0.4578 - loss: 1.4085 - val_accuracy: 0.4948 - val_loss: 1.3210 - learning_rate: 0.0010
Epoch 3/50
808/808 ━━━━━━━━━━━━━━━━━━━━ 0s 77ms/step - accuracy: 0.4799 - loss: 1.3642
Epoch 3: saving model to CNN_ep03.h5


808/808 ━━━━━━━━━━━━━━━━━━━━ 65s 80ms/step - accuracy: 0.4862 - loss: 1.3504 - val_accuracy: 0.5094 - val_loss: 1.2675 - learning_rate: 0.0010
Epoch 4/50
808/808 ━━━━━━━━━━━━━━━━━━━━ 0s 77ms/step - accuracy: 0.4919 - loss: 1.3223
Epoch 4: saving model to CNN_ep04.h5


808/808 ━━━━━━━━━━━━━━━━━━━━ 66s 81ms/step - accuracy: 0.4973 - loss: 1.3153 - val_accuracy: 0.5199 - val_loss: 1.2747 - learning_rate: 0.0010
Epoch 5/50
808/808 ━━━━━━━━━━━━━━━━━━━━ 0s 77ms/step - accuracy: 0.5136 - loss: 1.2872
Epoch 5: saving model to CNN_ep05.h5


...

808/808 ━━━━━━━━━━━━━━━━━━━━ 64s 79ms/step - accuracy: 0.6240 - loss: 1.0034 - val_accuracy: 0.6231 - val_loss: 1.0268 - learning_rate: 2.5000e-04
Epoch 47/50
808/808 ━━━━━━━━━━━━━━━━━━━━ 0s 76ms/step - accuracy: 0.6269 - loss: 0.9923
Epoch 47: saving model to CNN_ep47.h5



Epoch 47: ReduceLROnPlateau reducing learning rate to 0.0001250000059371814.
808/808 ━━━━━━━━━━━━━━━━━━━━ 65s 80ms/step - accuracy: 0.6227 - loss: 0.9992 - val_accuracy: 0.6140 - val_loss: 1.0192 - learning_rate: 2.5000e-04
Epoch 48/50
808/808 ━━━━━━━━━━━━━━━━━━━━ 0s 76ms/step - accuracy: 0.6241 - loss: 0.9951
Epoch 48: saving model to CNN_ep48.h5


808/808 ━━━━━━━━━━━━━━━━━━━━ 65s 81ms/step - accuracy: 0.6223 - loss: 0.9911 - val_accuracy: 0.6276 - val_loss: 1.0104 - learning_rate: 1.2500e-04
Epoch 49/50
808/808 ━━━━━━━━━━━━━━━━━━━━ 0s 74ms/step - accuracy: 0.6238 - loss: 0.9910
Epoch 49: saving model to CNN_ep49.h5


808/808 ━━━━━━━━━━━━━━━━━━━━ 80s 78ms/step - accuracy: 0.6282 - loss: 0.9836 - val_accuracy: 0.6241 - val_loss: 1.0192 - learning_rate: 1.2500e-04
Epoch 50/50
808/808 ━━━━━━━━━━━━━━━━━━━━ 0s 83ms/step - accuracy: 0.6356 - loss: 0.9710
Epoch 50: saving model to CNN_ep50.h5


808/808 ━━━━━━━━━━━━━━━━━━━━ 70s 87ms/step - accuracy: 0.6288 - loss: 0.9839 - val_accuracy: 0.6241 - val_loss: 1.0089 - learning_rate: 1.2500e-04
Restoring model weights from the end of the best epoch: 50.
